In [45]:
import cv2
import numpy as np
import smtplib
from google.colab import files
from tensorflow.keras.models import load_model
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.image import MIMEImage


In [46]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [47]:
model = load_model('/content/drive/MyDrive/project/aimodel.h5')

In [48]:
class_names = [
    "Abuse",
    "Arrest",
    "Assault",
    "Burglary",
    "Explosion",
    "Fighting",
    "RoadAccidents",
    "Robbery",
    "Shooting",
    "Shoplifting",
    "Stealing",
    "Vandalism"
]

In [49]:
suspicious_classes = [
    "Abuse",
    "Assault",
    "Burglary",
    "Explosion",
    "Fighting",
    "Robbery",
    "Shooting",
    "Shoplifting",
    "Stealing",
    "Vandalism"
]

In [50]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.image import MIMEImage

def send_email_alert(image_path, class_name):

    try:
        sender_email = "extra01dummy@gmail.com"
        receiver_email = "patelkavya8306@gmail.com"
        app_password = "vdcq xhqt flof qnzx"

        msg = MIMEMultipart()
        msg["From"] = sender_email
        msg["To"] = receiver_email
        msg["Subject"] = "🚨 CCTV ALERT: Suspicious Activity Detected"

        body = f"""
          ALERT!

          Suspicious activity detected:
          {class_name}

          Please take immediate action.
          """

        msg.attach(MIMEText(body, "plain"))

             # attach image
        with open(image_path, "rb") as f:
            img = MIMEImage(f.read())
            msg.attach(img)

        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(sender_email, app_password)
        server.sendmail(sender_email, receiver_email, msg.as_string())
        server.quit()

        print("✅ Email sent successfully")

    except Exception as e:
        print("❌ Email error:", e)

In [51]:
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

Saving Explosion004_x264_6.mp4 to Explosion004_x264_6.mp4


In [52]:
last_frame = None
alert_sent = False

In [53]:
cap = cv2.VideoCapture(video_path)

frames = []
alert_sent = False

print("🚀 Starting detection...")

while cap.isOpened():
  # Save last valid frame
    if frame is not None:
      last_frame = frame
    ret, frame = cap.read()

    if not ret:
        print("Video ended")
        break

    processed = preprocess(frame)
    frames.append(processed)

    # collect 4 frames
    if len(frames) == 4:

        data = np.array(frames).reshape(1, 4, 64, 64, 3)

        prediction = model.predict(data)

        predicted_class = int(np.argmax(prediction))
        confidence = float(np.max(prediction))

        class_name = class_names[predicted_class]

        print("Class:", class_name, "| Confidence:", confidence)

🚀 Starting detection...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step
Class: Explosion | Confidence: 0.9999998807907104
Video ended


In [54]:
if class_name in suspicious_classes and confidence > 0.5:
    print("⚠️ ALERT:", class_name)

    if not alert_sent:

        if last_frame is not None:
            try:
                cv2.imwrite("alert.jpg", last_frame)
                print("✅ Image saved")

                send_email_alert("alert.jpg", class_name)
                print("✅ Email sent")

                alert_sent = True

            except Exception as e:
                print("❌ Error saving/sending:", e)

        else:
            print("⚠️ No valid frame available")

print("✅ Finished")

⚠️ ALERT: Explosion
✅ Image saved
✅ Email sent successfully
✅ Email sent
✅ Finished


In [55]:
model.save('/content/drive/MyDrive/project/ai_mail_model.h5')